# IgLM -- Antibody Language Model

IgLM is a GPT-2 style antibody language model from the Gray Lab at Johns Hopkins, trained on antibody sequences from the Observed Antibody Space (OAS) with an *infilling* objective and conditioned on two control tokens, chain type (`heavy` / `light`) and species-of-origin (`human`, `mouse`, `rat`, `rabbit`, `rhesus`, `camel`). Because it learned to fill in a masked span rather than only to continue a prefix, one model covers three jobs: generating full-length antibodies from an N-terminal prompt (`run_iglm_sample`), re-designing an interior span such as CDR H3 while leaving the framework intact (`run_iglm_infill`), and scoring how antibody-like a whole sequence or a single designed span is (`run_iglm_score`) -- the design-then-filter loop this notebook walks through end to end.

**Links:** [Paper -- Shuai, Ruffolo & Gray, *Cell Systems* (2023)](https://doi.org/10.1016/j.cels.2023.10.001) | [GitHub](https://github.com/Graylab/IgLM)

> **License:** IgLM is released under the JHU Academic Software License Agreement -- **non-commercial use only**. Review the license before using the model or its weights in a commercial setting.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("iglm")
display_overview("iglm")
display_docs_section("iglm", "Background")

# IgLM

[IgLM](https://github.com/Graylab/IgLM) is an antibody-specific generative language model from the Gray Lab at Johns Hopkins University, trained on 558 million antibody variable-domain sequences from the [Observed Antibody Space](https://opig.stats.ox.ac.uk/webapps/oas/). Its infilling objective lets it re-design a variable-length span inside an antibody using the residues on both sides as context, and every generation is conditioned on chain type and species of origin. This toolkit exposes full-length generation, span re-design, and likelihood scoring.

IgLM ([Shuai et al., 2023](https://doi.org/10.1016/j.cels.2023.10.001)) is a [GPT-2](https://en.wikipedia.org/wiki/GPT-2) decoder trained on 558 million antibody heavy- and light-chain variable sequences: the 809 million unique sequences then available in the [Observed Antibody Space](https://opig.stats.ox.ac.uk/webapps/oas/) (OAS) were clustered at 95% sequence identity down to 588 million non-redundant sequences, a random 5% of which was held out for testing, leaving 558 million for training. Every training sequence is prefixed with two conditioning tags naming its chain type (heavy or light) and its species of origin (human, mouse, rat, rabbit, rhesus, or camel), so the same checkpoint generates and scores across chains and species. Two checkpoints are released: `IgLM`, the model used throughout the paper, with 4 layers, 512-dimensional embeddings, 8 attention heads, and 12,889,600 parameters; and `IgLM-S`, a smaller variant with 3 layers, 192-dimensional embeddings, 6 attention heads, and 1,439,616 parameters. Both released checkpoints share a 33-token vocabulary (the 20 amino acids, the two chain tags, the six species tags, and five special tokens) and a 512-token context window.

What distinguishes IgLM from a left-to-right protein language model is its training objective, which the paper formulates as text infilling. For each training sequence a span of length drawn uniformly from 10 to 20 residues is replaced in place by a mask token and then re-appended after a separator token, so the model learns to emit the span conditioned on the residues that both precede *and* follow it while remaining an ordinary autoregressive decoder. That matters because the [complementarity-determining regions](https://en.wikipedia.org/wiki/Complementarity-determining_region) that dominate antigen binding sit in the middle of the variable domain, between conserved framework regions: re-designing one means generating into a gap with known flanks, not continuing a prefix. On a 30-million-sequence held-out test set, IgLM with bidirectional context achieved lower infilling perplexity on CDR loops than the same model restricted to preceding context and than ProGen2-base or ProGen2-OAS, which the authors read as evidence that aligning the pre-training objective with the design task is what buys the improvement.

The paper's headline application is CDR loop library design. Infilling the CDR H3 loop of each of 49 therapeutic antibodies across sampling temperatures of 0.8, 1.0, and 1.2 and nucleus probabilities of 0.5, 0.75, and 1.0 produced libraries whose members showed reduced predicted aggregation propensity, increased predicted solubility (CamSol Intrinsic), and higher OASis identity - the fraction of a sequence's 9-mers seen in human repertoires - than their parent antibodies. The same model likelihoods also carry signal on their own: used to classify sequences from the IMGT mAb DB as human versus non-human, IgLM reached a ROC AUC of 0.96, ahead of ProGen2-OAS at 0.94, which is the basis for treating IgLM likelihood as a naturalness score for antibody sequences.

## Available tools

In [2]:
display_available_tools("iglm")

- **`run_iglm_infill()`** — Re-design a span of an antibody sequence with IgLM, conditioned on the rest of the chain
- **`run_iglm_sample()`** — Generate full-length antibody sequences with IgLM, conditioned on chain and species
- **`run_iglm_score()`** — Score antibody sequences with IgLM, conditioned on chain and species

## Imports

In [3]:
from proto_tools.tools import (
    IgLMInfillConfig,
    IgLMInfillInput,
    IgLMSampleConfig,
    IgLMSampleInput,
    IgLMScoringConfig,
    IgLMScoringInput,
    IgLMSpan,
    run_iglm_infill,
    run_iglm_sample,
    run_iglm_score,
)

## API Reference

### `run_iglm_sample`

Generates full-length antibody sequences by continuing an N-terminal prompt from the chain and species conditioning tokens. `prompts`, `chains`, and `species` are parallel, broadcastable fields: pass one chain/species to condition every prompt the same way, or one value per prompt to mix chains and species in a single call.

In [4]:
display_api_reference("iglm", "input", "run_iglm_sample")

**Input** — `IgLMSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>chains</code> | <code>list[Literal['heavy', 'light']]</code> | <code>['heavy']</code> | Chain type per item; one value conditions every item the same way |
| <code>species</code> | <code>list[Literal['human', 'mouse', 'rat', 'rabbit', 'rhesus', 'camel']]</code> | <code>['human']</code> | Species-of-origin per item; one value conditions every item the same way |
| <code>prompts</code> | <code>list[str]</code> | <code>['']</code> | N-terminal prefix per item; an empty string generates from scratch |

In [5]:
display_api_reference("iglm", "config", "run_iglm_sample")

**Config** — `IgLMSampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>model_checkpoint</code> | <code>Literal['IgLM', 'IgLM-S']</code> | <code>'IgLM'</code> | IgLM weights variant; 'IgLM' is 13M parameters, 'IgLM-S' is 1.4M |
| <code>num_sequences_per_input</code> | <code>int</code> | <code>1</code> | Distinct sequences to generate for each input item |
| <code>temperature</code> | <code>float</code> | <code>1.0</code> | Softmax temperature for sampling; lower is more deterministic |
| <code>top_p</code> | <code>float</code> | <code>1.0</code> | Nucleus sampling cutoff over per-position token probabilities |
| <code>max_length</code> | <code>int</code> | <code>150</code> | Token budget per generation, including conditioning tokens and prompt |
| <code>batch_size</code> | <code>int</code> | <code>8</code> | Candidate sequences per GPU forward pass; raise for throughput, lower if OOM |
| <code>prepend_prompt</code> | <code>bool</code> | <code>True</code> | Keep the prompt at the start of each generated sequence |

In [6]:
display_api_reference("iglm", "output", "run_iglm_sample")

**Output** — `IgLMGenerationOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[IgLMSamples]</code> | required | Generated sequences bundled per input item, in input order |

**`IgLMSamples`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Distinct sequences generated for this input, in generation order |

### `run_iglm_infill`

Replaces a residue span with a mask token and lets IgLM write a new segment conditioned on the surrounding residues -- the objective it was trained on. The new segment is **not** constrained to the original span length, so each returned bundle reports `infilled_residues` (what IgLM wrote) alongside `spans` (where those residues sit in the returned sequence), ready to hand straight back to `run_iglm_score`. Spans are 1-indexed and inclusive, and each parent sequence needs exactly one span.

In [7]:
display_api_reference("iglm", "input", "run_iglm_infill")
display_api_reference("iglm", "config", "run_iglm_infill")
display_api_reference("iglm", "output", "run_iglm_infill")

**Input** — `IgLMInfillInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>chains</code> | <code>list[Literal['heavy', 'light']]</code> | <code>['heavy']</code> | Chain type per item; one value conditions every item the same way |
| <code>species</code> | <code>list[Literal['human', 'mouse', 'rat', 'rabbit', 'rhesus', 'camel']]</code> | <code>['human']</code> | Species-of-origin per item; one value conditions every item the same way |
| <code>sequences</code> | <code>list[str]</code> | required | Parent antibody sequence(s) whose span is re-designed |
| <code>spans</code> | <code>list[IgLMSpan]</code> | required | Residue span re-designed in each parent sequence (1-indexed, inclusive) |

**Config** — `IgLMInfillConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>model_checkpoint</code> | <code>Literal['IgLM', 'IgLM-S']</code> | <code>'IgLM'</code> | IgLM weights variant; 'IgLM' is 13M parameters, 'IgLM-S' is 1.4M |
| <code>num_sequences_per_input</code> | <code>int</code> | <code>1</code> | Distinct sequences to generate for each input item |
| <code>temperature</code> | <code>float</code> | <code>1.0</code> | Softmax temperature for sampling; lower is more deterministic |
| <code>top_p</code> | <code>float</code> | <code>1.0</code> | Nucleus sampling cutoff over per-position token probabilities |
| <code>max_length</code> | <code>int</code> | <code>150</code> | Token budget per generation, including conditioning tokens and prompt |
| <code>batch_size</code> | <code>int</code> | <code>8</code> | Candidate sequences per GPU forward pass; raise for throughput, lower if OOM |

**Output** — `IgLMInfillOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[IgLMInfilledSequences]</code> | required | Re-designed sequences bundled per parent sequence, in input order |

**`IgLMInfilledSequences`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Distinct sequences generated for this input, in generation order |
| <code>infilled_residues</code> | <code>list[str]</code> | required | Residues written into the span, aligned with 'sequences' |
| <code>spans</code> | <code>list[IgLMSpan]</code> | required | Where the written residues sit in each sequence (1-indexed, inclusive) |

**`IgLMSpan`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>start</code> | <code>int</code> | required | First residue of the span (1-indexed, inclusive) |
| <code>end</code> | <code>int</code> | required | Last residue of the span (1-indexed, inclusive) |

### `run_iglm_score`

Computes the autoregressive likelihood of each sequence under its chain and species conditioning. Supplying a `span` switches to IgLM's infilling likelihood: the span is scored given the rest of the chain, which is how designed CDR loops are ranked against each other. Omitting `spans` (or passing `None`) scores the full sequence.

In [8]:
display_api_reference("iglm", "input", "run_iglm_score")
display_api_reference("iglm", "config", "run_iglm_score")
display_api_reference("iglm", "output", "run_iglm_score")

**Input** — `IgLMScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>chains</code> | <code>list[Literal['heavy', 'light']]</code> | <code>['heavy']</code> | Chain type per item; one value conditions every item the same way |
| <code>species</code> | <code>list[Literal['human', 'mouse', 'rat', 'rabbit', 'rhesus', 'camel']]</code> | <code>['human']</code> | Species-of-origin per item; one value conditions every item the same way |
| <code>sequences</code> | <code>list[str]</code> | required | Sequences to score |
| <code>spans</code> | <code>list[IgLMSpan &#124; None] &#124; None</code> | <code>None</code> | Span to score conditionally per sequence (1-indexed, inclusive); None scores all |

**Config** — `IgLMScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>batch_size</code> | <code>int</code> | <code>8</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include per-position logits in the output (large; disable to save memory) |
| <code>model_checkpoint</code> | <code>Literal['IgLM', 'IgLM-S']</code> | <code>'IgLM'</code> | IgLM weights variant; 'IgLM' is 13M parameters, 'IgLM-S' is 1.4M |

**Output** — `CausalModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>scores</code> | <code>list[CausalModelScoringMetrics]</code> | required | List of scoring outputs, one per input sequence |

**Metrics** — `CausalModelScoringMetrics` (one set per `scores` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>log_likelihood</code> | <code>float</code> | <code>&lt;= 0</code> |  | Sum over all positions. Grows with sequence length, so compare only at equal length. |
| <code>avg_log_likelihood</code> | <code>float</code> | <code>&lt;= 0</code> |  | Mean per position. Length independent, so comparable across sequences. |
| <code>perplexity</code> **(primary)** | <code>float</code> | <code>&gt;= 1</code> |  | Exponential of the negated mean log-likelihood. Lower means a more confident model. |

## 1. Generate full-length antibody sequences

IgLM resamples until it holds `num_sequences_per_input` *distinct* sequences per prompt, discarding malformed generations the same way the upstream implementation does. Generation stops at IgLM's end-of-sequence token or at `max_length` tokens, which counts the two conditioning tokens and the prompt.

In [9]:
# Five distinct human heavy chains continuing the N-terminal prompt "EVQ".
# One chain/species value conditions every prompt the same way (broadcasting).
sample_inputs = IgLMSampleInput(prompts=["EVQ"], chains=["heavy"], species=["human"])

sample_config = IgLMSampleConfig(
    model_checkpoint="IgLM",      # 13M-parameter model from the paper ("IgLM-S" is 1.4M)
    num_sequences_per_input=5,    # Distinct sequences per prompt
    temperature=1.0,
    top_p=0.95,
    prepend_prompt=True,          # Keep "EVQ" at the start of each sequence
    batch_size=8,
    seed=0,                       # Reproducible sampling
    device="cuda",                # Change to "cpu" if no GPU available
)

sample_result = run_iglm_sample(sample_inputs, sample_config)

# One IgLMSamples bundle per prompt; each bundle is iterable and indexable.
heavy_bundle = sample_result.results[0]
print(f"Generated {len(heavy_bundle)} distinct human heavy chains from the prompt 'EVQ':")
for i, sequence in enumerate(heavy_bundle, start=1):
    print(f"  {i}. ({len(sequence)} aa) {sequence}")

Running run_iglm_sample [00:00]

Generated 5 distinct human heavy chains from the prompt 'EVQ':
  1. (124 aa) EVQLVESGGGLVQPGGSLRLSCAASGFTFSSYWMHWVRQAPGKGLVWVSRINSDGSSTSYADSVKGRFTISRDNAKNTLYLQMNSLRAEDTAVYYCARTYSSSWSYYYYGMDVWGQGTTVTVSS
  2. (122 aa) EVQLLESGGGLVQPGGSLRLSCAASGFTFSSYAMSWVRQAPGKGLEWVSAISGSGGSTYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAKDFGYSYRADAFDIWGQGTMVTVSS
  3. (123 aa) EVQLVESGGGLVKPGGSLRLSCAASGFSFSNAWMNWVRQTPGKGLEWVGRIKTIGDGGTTDYAAPVKGRFTISRDDSKNTLYLQMNSLKTEDTAVYYCATHLPVATGRGFDCWGQGTLVTVSS
  4. (120 aa) EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWMSWVRQAPGKGLEWVGRIKSKTDGGTTDYAAPVKGRFTISRDDSKNTLYLQMNSLKTEDTAVYYCTTALRSGRLDYWGQGTLVTVSS
  5. (121 aa) EVQLVQSGAEVKKPGESLKISCKGSGYSFTSYWIGWVRQMPGKGLEWMGIIYPGDSDTRYSPSFQGQVTISADKSISTAYLQWSSLKASDTAMYYCARQRRIAAAGYFDYWGQGTLVTVSS


### Nanobodies: the same call with camel conditioning

Swapping the species token to `camel` biases generation toward camelid single-domain heavy chains (VHHs / nanobodies), which conventionally start `QVQ` rather than `EVQ`.

In [10]:
# Camel heavy chains (nanobodies) from the prompt "QVQ".
nanobody_inputs = IgLMSampleInput(prompts=["QVQ"], chains=["heavy"], species=["camel"])
nanobody_config = IgLMSampleConfig(
    num_sequences_per_input=3,
    temperature=1.0,
    top_p=0.95,
    seed=0,
    device="cuda",  # Change to "cpu" if no GPU available
)

nanobody_result = run_iglm_sample(nanobody_inputs, nanobody_config)

print("Camel heavy chains (nanobodies) from the prompt 'QVQ':")
for i, sequence in enumerate(nanobody_result.results[0], start=1):
    print(f"  {i}. ({len(sequence)} aa) {sequence}")

Running run_iglm_sample [00:00]

Camel heavy chains (nanobodies) from the prompt 'QVQ':
  1. (114 aa) QVQLVESGGGSVEAGGSLRLSCAASRYIFGSYCMAWFRQAPGKEREGVAAIDSDGSTSYADSVKGRFTISKDNAKNTLYLQMNSLKPEDTAMYYCATGDGAGLYWGQGTQVTVS
  2. (115 aa) QVQLVESGGGLVQPGGSLRLSCVVSGDTFSLYWMYWVRQAPGKGLEWVSRINGGGGSTYYADSVKGRFTISRDNAKNTLYLQLNSLKTEDTAMYYCTKNLYSPDYWGQGTQVTVS
  3. (122 aa) QVQLVESGGGSVQSGGSLRLSCAASGYSRSNNCMAWFRQGPGKEREGVAAIDTDGTTNYADSVKGRFTISKDNAKKTLYLQMNSLKPEDTAMYYCAAGVWDVTYSATETFDSWGQGTQVTVS


## 2. Re-design a CDR loop by infilling

The usual engineering task is not generating a whole new antibody but diversifying one loop of an existing one. Here we mask CDR H3 of the 1JPT heavy chain (residues 99-106, the loop `DTAAYFDY`) and let IgLM propose replacements conditioned on the untouched framework.

In [11]:
# 1JPT heavy chain; residues 99-106 are its CDR H3 loop.
parent_vh = (
    "EVQLVESGGGLVQPGGSLRLSCAASGFNIKEYYMHWVRQAPGKGLEWVGLIDPEQGNTIYDPKFQDRATIS"
    "ADNSKNTAYLQMNSLRAEDTAVYYCARDTAAYFDYWGQGTLVTVS"
)

# Spans are 1-indexed and inclusive, so slicing back out needs start - 1.
cdr_h3 = IgLMSpan(start=99, end=106)
original_loop = parent_vh[cdr_h3.start - 1 : cdr_h3.end]
print(f"Parent heavy chain: {len(parent_vh)} aa")
print(f"Original CDR H3:    {original_loop} ({len(original_loop)} aa)")

Parent heavy chain: 116 aa
Original CDR H3:    DTAAYFDY (8 aa)


In [12]:
# One span per parent sequence; chain/species broadcast as in generation.
infill_inputs = IgLMInfillInput(
    sequences=[parent_vh],
    spans=[cdr_h3],
    chains=["heavy"],
    species=["human"],
)

infill_config = IgLMInfillConfig(
    num_sequences_per_input=6,  # Six distinct re-designs of the loop
    temperature=1.0,
    top_p=0.95,
    batch_size=8,
    seed=0,
    device="cuda",  # Change to "cpu" if no GPU available
)

infill_result = run_iglm_infill(infill_inputs, infill_config)

# The bundle carries three aligned lists: the full re-designed chains, the residues
# IgLM wrote, and the span each new segment occupies in its returned sequence.
designs = infill_result.results[0]

print(f"Original CDR H3: {original_loop} ({len(original_loop)} aa)")
print("Designed CDR H3 loops (IgLM chooses the length, so they need not match):")
for i, (loop, span, design) in enumerate(
    zip(designs.infilled_residues, designs.spans, designs.sequences), start=1
):
    print(f"  design {i}: {loop:<16} ({len(loop)} aa at {span.start}-{span.end}, full chain {len(design)} aa)")

Running run_iglm_infill [00:00]

Original CDR H3: DTAAYFDY (8 aa)
Designed CDR H3 loops (IgLM chooses the length, so they need not match):
  design 1: DTNWNKSYYFDY     (12 aa at 99-110, full chain 120 aa)
  design 2: DFCGGCDYRVGY     (12 aa at 99-110, full chain 120 aa)
  design 3: SILVGSNAVDS      (11 aa at 99-109, full chain 119 aa)
  design 4: DADTSSYHSLFAY    (13 aa at 99-111, full chain 121 aa)
  design 5: LNSGHYPLDY       (10 aa at 99-108, full chain 118 aa)
  design 6: ADYDFWGGSNWFDP   (14 aa at 99-112, full chain 122 aa)


## 3. Score and rank the designs

This is the payoff of the generate-then-filter loop. We first score the unmodified parent as a
whole chain, to show what whole-sequence scoring looks like, and then re-score every design
*conditioned on its span*: IgLM reports the likelihood of just the new loop given the rest of the
chain, which is the quantity to rank CDR designs by. The two numbers are not comparable with each
other - one covers 117 tokens and the others cover a loop apiece - so the wild-type loop is carried
into the span-conditional call below to serve as the baseline for the ranking. Because the infill
output already reports `spans`, the designs hand straight back to `run_iglm_score` with no index
arithmetic.

In [13]:
# Whole-sequence scoring: no spans, so the full chain is scored.
parent_score_inputs = IgLMScoringInput(
    sequences=[parent_vh],
    chains=["heavy"],
    species=["human"],
)
parent_score_result = run_iglm_score(
    parent_score_inputs,
    IgLMScoringConfig(batch_size=8, device="cuda"),  # Change device to "cpu" if no GPU available
)

parent_metrics = parent_score_result.scores[0]
print("Parent 1JPT heavy chain, scored as a human heavy chain:")
print(f"    Log-likelihood:     {parent_metrics.log_likelihood:.3f}")
print(f"    Avg log-likelihood: {parent_metrics.avg_log_likelihood:.3f}")
print(f"    Perplexity:         {parent_metrics.perplexity:.3f}  (lower = more antibody-like)")

Running run_iglm_score [00:00]

Parent 1JPT heavy chain, scored as a human heavy chain:
    Log-likelihood:     -132.141
    Avg log-likelihood: -1.129
    Perplexity:         3.094  (lower = more antibody-like)


In [14]:
# Span-conditional scoring: pass the designs' own spans straight through.
# Scoring the wild-type loop in the same call gives the designs a baseline.
candidate_sequences = [parent_vh, *designs.sequences]
candidate_spans = [cdr_h3, *designs.spans]
candidate_loops = [original_loop, *designs.infilled_residues]
candidate_labels = ["wild type", *(f"design {i}" for i in range(1, len(designs.sequences) + 1))]

rank_inputs = IgLMScoringInput(
    sequences=candidate_sequences,
    spans=candidate_spans,
    chains=["heavy"],
    species=["human"],
)
rank_result = run_iglm_score(
    rank_inputs,
    IgLMScoringConfig(batch_size=8, device="cuda"),  # Change device to "cpu" if no GPU available
)

Running run_iglm_score [00:00]

In [15]:
# Rank by avg_log_likelihood (and its monotone partner, perplexity): the loops
# differ in length, so the summed log_likelihood is not comparable across candidates.
ranked = sorted(
    zip(candidate_labels, candidate_loops, rank_result.scores),
    key=lambda candidate: candidate[2].avg_log_likelihood,
    reverse=True,
)

print(f"{'rank':<6}{'candidate':<12}{'CDR H3':<18}{'avg log-lik':>12}{'perplexity':>13}")
for rank, (label, loop, score) in enumerate(ranked, start=1):
    print(f"{rank:<6}{label:<12}{loop:<18}{score.avg_log_likelihood:>12.3f}{score.perplexity:>13.3f}")

best_label, best_loop, best_score = ranked[0]
print(f"\nBest-scoring loop: {best_loop} ({best_label}), perplexity {best_score.perplexity:.3f}")
print("Scores are only comparable under identical conditioning (same chain and species tokens).")

rank  candidate   CDR H3             avg log-lik   perplexity
1     design 1    DTNWNKSYYFDY            -0.871        2.389
2     design 4    DADTSSYHSLFAY           -1.120        3.063
3     design 6    ADYDFWGGSNWFDP          -1.301        3.672
4     design 2    DFCGGCDYRVGY            -1.713        5.546
5     design 5    LNSGHYPLDY              -1.757        5.796
6     wild type   DTAAYFDY                -2.031        7.625
7     design 3    SILVGSNAVDS             -2.395       10.965

Best-scoring loop: DTNWNKSYYFDY (design 1), perplexity 2.389
Scores are only comparable under identical conditioning (same chain and species tokens).


## 4. Export results

Generation and infilling results export to `fasta`, `txt`, or `json`; scoring results export to `csv` or `json`.

In [16]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Generated sequences -> FASTA
sample_result.export(name="iglm_heavy_chains", export_path=output_dir, file_format="fasta")
print(f"Generated heavy chains exported to {output_dir / 'iglm_heavy_chains.fasta'}")

infill_result.export(name="iglm_cdr_h3_designs", export_path=output_dir, file_format="fasta")
print(f"CDR H3 re-designs exported to {output_dir / 'iglm_cdr_h3_designs.fasta'}")

# Scoring metrics -> CSV
rank_result.export(name="iglm_design_scores", export_path=output_dir, file_format="csv")
print(f"Design scores exported to {output_dir / 'iglm_design_scores.csv'}")

Generated heavy chains exported to example_output/iglm_heavy_chains.fasta
CDR H3 re-designs exported to example_output/iglm_cdr_h3_designs.fasta
Design scores exported to example_output/iglm_design_scores.csv
